# 🎨 AbiArt — Kaggle GPU Backend (Automatic1111 SD WebUI API)

### ⚡ Avant de lancer :
1. **Accelerator** → `GPU T4 x2` (panneau à droite)
2. **Internet** → `ON`
3. Cliquez sur **Run All**
4. Copiez l'URL Cloudflare affichée à la fin et collez-la dans AbiArt → Kaggle → Connect

In [ ]:
# ══════════════════════════════════════════════════
# CELLULE 1 — GPU CHECK
# ══════════════════════════════════════════════════
import subprocess, sys
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if r.returncode != 0:
    print('\u274c ERREUR : Aucun GPU détecté !')
    print('    → Allez dans le panneau à droite → Accelerator → GPU T4 x2')
    print('    → Puis relancez ce notebook.')
    sys.exit(1)
else:
    print(r.stdout)
    print('\u2713 GPU détecté !')

In [ ]:
# ══════════════════════════════════════════════════
# CELLULE 2 — CLONE SD WEBUI + PATCH + INSTALL ALL DEPS
# ══════════════════════════════════════════════════
import os
WEBUI = '/kaggle/working/stable-diffusion-webui'
os.environ['STABLE_DIFFUSION_REPO'] = 'https://github.com/w-e-w/stablediffusion.git'

# --- Clone WebUI ---
if not os.path.exists(WEBUI):
    !git clone https://github.com/AUTOMATIC1111/stable-diffusion-webui.git {WEBUI}
%cd {WEBUI}

# --- Patch UNIQUEMENT les 2 paquets qui ne compilent pas sous Python 3.11+ ---
!sed -i 's/Pillow==9.5.0/Pillow>=10.0.0/g' requirements_versions.txt
!sed -i 's/scikit-image==0.21.0/scikit-image>=0.21.0/g' requirements_versions.txt

# --- Installer TOUTES les dépendances pip avec progression visible ---
print('\n[1/4] Installation des requirements A1111...')
!pip install -r requirements_versions.txt --prefer-binary 2>&1 | tail -5

print('\n[2/4] Installation de CLIP (--no-build-isolation)...')
!pip install --no-build-isolation https://github.com/openai/CLIP/archive/d50d76daa670286dd6cacf3bcd80b5e4823fc8e1.zip 2>&1 | tail -3

print('\n[3/4] Installation d\'open-clip-torch et extensions...')
!pip install -q open-clip-torch>=2.20.0 xformers ultralytics mediapipe

# --- Clone TOUS les sous-dépôts (launch.py --skip-install ne les clone PAS) ---
print('\n[4/4] Clonage des sous-dépôts...')
!mkdir -p repositories
repos = {
    'stable-diffusion-stability-ai': 'https://github.com/w-e-w/stablediffusion.git',
    'generative-models':             'https://github.com/Stability-AI/generative-models.git',
    'k-diffusion':                   'https://github.com/crowsonkb/k-diffusion.git',
    'BLIP':                          'https://github.com/salesforce/BLIP.git',
    'stable-diffusion-webui-assets': 'https://github.com/AUTOMATIC1111/stable-diffusion-webui-assets.git',
}
for name, url in repos.items():
    dest = f'repositories/{name}'
    if not os.path.exists(dest):
        print(f'  Cloning {name}...')
        !git clone --depth 1 {url} {dest} 2>/dev/null
    else:
        print(f'  {name} déjà présent')

# --- Extensions ---
!git clone --depth 1 https://github.com/Bing-su/adetailer.git extensions/adetailer 2>/dev/null || true
!git clone --depth 1 https://github.com/Mikubill/sd-webui-controlnet.git extensions/sd-webui-controlnet 2>/dev/null || true

print('\n\u2713 Installation terminée !')

In [ ]:
# ══════════════════════════════════════════════════
# CELLULE 3 — VÉRIFICATION : tous les modules critiques
# ══════════════════════════════════════════════════
import importlib, sys

modules_critiques = [
    ('torch',                          'PyTorch'),
    ('torchvision',                    'TorchVision'),
    ('xformers',                       'xFormers'),
    ('PIL',                            'Pillow'),
    ('skimage',                        'scikit-image'),
    ('open_clip',                      'open-clip-torch'),
    ('clip',                           'CLIP (OpenAI)'),
    ('pytorch_lightning',              'PyTorch Lightning'),
    ('pytorch_lightning.utilities.distributed', 'PL distributed (v1.9.4)'),
    ('gradio',                         'Gradio'),
    ('transformers',                   'Transformers'),
    ('safetensors',                    'Safetensors'),
    ('accelerate',                     'Accelerate'),
    ('einops',                         'Einops'),
    ('kornia',                         'Kornia'),
    ('omegaconf',                      'OmegaConf'),
    ('ldm',                            'LDM (stable-diffusion)'),
    ('sgm',                            'SGM (generative-models)'),
    ('k_diffusion',                    'k-diffusion'),
]

# Ajouter repositories/ au path pour ldm, sgm, etc.
import os
WEBUI = '/kaggle/working/stable-diffusion-webui'
for sub in ['stable-diffusion-stability-ai', 'generative-models', 'k-diffusion', 'BLIP']:
    p = os.path.join(WEBUI, 'repositories', sub)
    if p not in sys.path:
        sys.path.insert(0, p)

ok, fail = 0, 0
for mod, label in modules_critiques:
    try:
        importlib.import_module(mod)
        print(f'  \u2713 {label}')
        ok += 1
    except Exception as e:
        print(f'  \u274c {label}  →  {e}')
        fail += 1

print(f'\n{ok} OK / {fail} ERREUR(S)')
if fail > 0:
    print('\u26a0\ufe0f  Des modules manquent. Vérifiez les erreurs ci-dessus avant de continuer.')
else:
    print('\u2713 Tous les modules sont prêts ! Vous pouvez lancer la cellule suivante.')

In [ ]:
# ══════════════════════════════════════════════════
# CELLULE 4 — MODÈLES (Animagine XL 3.1 + SDXL VAE)
# ══════════════════════════════════════════════════
import os
WEBUI = '/kaggle/working/stable-diffusion-webui'
ckpt_dir = f'{WEBUI}/models/Stable-diffusion'
vae_dir  = f'{WEBUI}/models/VAE'

os.makedirs(ckpt_dir, exist_ok=True)
os.makedirs(vae_dir, exist_ok=True)

if not os.path.exists(f'{ckpt_dir}/animagine-xl-3.1.safetensors'):
    print('Téléchargement Animagine XL 3.1...')
    !wget -q --show-progress -c https://huggingface.co/cagliostrolab/animagine-xl-3.1/resolve/main/animagine-xl-3.1.safetensors -O {ckpt_dir}/animagine-xl-3.1.safetensors
else:
    print('Animagine XL 3.1 déjà présent')

if not os.path.exists(f'{vae_dir}/sdxl_vae.safetensors'):
    print('Téléchargement SDXL VAE...')
    !wget -q --show-progress -c https://huggingface.co/stabilityai/sdxl-vae/resolve/main/sdxl_vae.safetensors -O {vae_dir}/sdxl_vae.safetensors
else:
    print('SDXL VAE déjà présent')

print('\u2713 Modèles prêts !')

In [ ]:
# ══════════════════════════════════════════════════
# CELLULE 5 — CLOUDFLARE TUNNEL + LANCEMENT API
# ══════════════════════════════════════════════════
import os, subprocess, time, re, threading

WEBUI = '/kaggle/working/stable-diffusion-webui'
os.environ['STABLE_DIFFUSION_REPO'] = 'https://github.com/w-e-w/stablediffusion.git'

# Installer Cloudflare Tunnel
!wget -q -c https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb >/dev/null 2>&1

# Démarrer le tunnel
tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:7860'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
)

def show_url():
    for line in iter(tunnel.stderr.readline, ''):
        m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if m:
            print('\n' + '='*60)
            print('🎨 URL KAGGLE POUR ABIART :')
            print(f'   >>>  {m.group(0)}  <<<')
            print('='*60)
            print('Collez cette URL dans AbiArt → Backend Kaggle → Connect\n')
            break

threading.Thread(target=show_url, daemon=True).start()
time.sleep(3)

# Lancement IMMDIAT avec --skip-install (tout est déjà installé)
%cd {WEBUI}
!python launch.py \
    --skip-install \
    --skip-torch-cuda-test \
    --enable-insecure-extension-access \
    --no-half-vae \
    --xformers \
    --api \
    --listen \
    --port 7860